# OpenFold Geometric Stability Scaling Experiment

Tests geometric stability of the **OpenFold Evoformer** (AlphaFold2 reproduction) across increasing protein sequence lengths using **SoloSeq mode** (single-sequence, no MSA required).

**Scaling variable**: Protein sequence length (100, 200, 400, 800 AA)  
Unlike ESM-2 (which scales by model size), OpenFold has one model size. We test whether the Evoformer's geometric processing degrades with longer inputs.

**Pipeline**: `AA sequence → ESM-1b embeddings → OpenFold Evoformer → Single Representation (c_s=384) → Mean Pool → StabilityHarness`

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py` to this Colab session
2. Change Runtime to **GPU** (Runtime > Change runtime type > GPU, preferably A100 for memory)
3. Run cells in order

**Note**: SoloSeq is limited to sequences of 1022 residues or fewer (ESM-1b constraint). Both ESM-1b (~2.5 GB) and OpenFold (~1.5 GB) must fit in GPU memory simultaneously.

---

In [ ]:
OPENFOLD_DIR = os.environ.get('OPENFOLD_DIR', '/content/openfold')
WEIGHTS_DIR = os.environ.get('OF_WEIGHTS_DIR', './results/openfold_weights')

# Install Dependencies
import os, sys

print("Installing base dependencies...")
!pip install -q torch matplotlib seaborn pandas

print("Installing ESM (Facebook)...")
!pip install -q fair-esm

print("Installing shesha-geometry...")
!pip install -q shesha-geometry

!pip install -q biopython 
!pip install ml-collections
!pip install modelcif
!pip install -q awscli

print("Installing OpenFold...")
try:
    import openfold
    print(f"  OpenFold already installed: {openfold.__file__}")
except ImportError:
    !pip install -q "openfold @ git+https://github.com/aqlaboratory/openfold.git" 2>/dev/null
    try:
        import openfold
        print(f"  OpenFold installed via pip: {openfold.__file__}")
    except ImportError:
        if not os.path.exists(OPENFOLD_DIR):
            print("  Cloning OpenFold repo...")
            !git clone --depth 1 -q https://github.com/aqlaboratory/openfold.git {OPENFOLD_DIR}
        sys.path.insert(0, OPENFOLD_DIR)
        try:
            import openfold
            print(f"  OpenFold loaded from clone: {openfold.__file__}")
        except ImportError as e:
            print(f"  ERROR: Could not install OpenFold: {e}")
            print("  Try manually: pip install git+https://github.com/aqlaboratory/openfold.git")

# Download SoloSeq weights (with size check and subfolder handling)
WEIGHTS_DIR = WEIGHTS_DIR
WEIGHTS_FILE = 'seq_model_esm1b_ptm.pt'
WEIGHTS_PATH = f'{WEIGHTS_DIR}/{WEIGHTS_FILE}'
MIN_WEIGHTS_SIZE_BYTES = 50_000_000  # ~50 MB; smaller usually means incomplete/corrupt

        "def _ensure_soloseq_weights():\n",
        "    global WEIGHTS_PATH\n",
        "    if os.path.exists(WEIGHTS_PATH):\n",
        "        size = os.path.getsize(WEIGHTS_PATH)\n",
        "        if size >= MIN_WEIGHTS_SIZE_BYTES:\n",
        "            return True\n",
        "        print(f"Removing incomplete weights ({size / 1e6:.0f} MB)...")\n",
        "        os.remove(WEIGHTS_PATH)\n",
        "    os.makedirs(WEIGHTS_DIR, exist_ok=True)\n",
        "    print("\\nDownloading SoloSeq weights via AWS S3...")\n",
        "    import subprocess, shutil\n",
        "    # Direct S3 download (no credentials needed)\n",
        "    dl_dir = f'{WEIGHTS_DIR}/openfold_soloseq_params'\n",
        "    os.makedirs(dl_dir, exist_ok=True)\n",
        "    subprocess.run([\n",
        "        'aws', 's3', 'cp', '--no-sign-request', '--region', 'us-east-1',\n",
        "        's3://openfold/openfold_soloseq_params/', dl_dir, '--recursive'\n",
        "    ], check=False)\n",
        "    # Move weights file to expected location\n",
        "    for root, _, files in os.walk(WEIGHTS_DIR):\n",
        "        for f in files:\n",
        "            if f == WEIGHTS_FILE:\n",
        "                src = os.path.join(root, f)\n",
        "                if os.path.abspath(src) != os.path.abspath(WEIGHTS_PATH):\n",
        "                    print(f"  Copying {src} -> {WEIGHTS_PATH}")\n",
        "                    shutil.copy2(src, WEIGHTS_PATH)\n",
        "                break\n",
        "    return os.path.exists(WEIGHTS_PATH) and os.path.getsize(WEIGHTS_PATH) >= MIN_WEIGHTS_SIZE_BYTES\n",
if _ensure_soloseq_weights():
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f"\nSoloSeq weights ready: {WEIGHTS_PATH} ({size_mb:.0f} MB)")
else:
    print(f"\nWARNING: Weights not found or incomplete at {WEIGHTS_PATH}")

print("\nDone!")

In [ ]:
# Configuration
import os

PHASE = 'quick'  # Start here!

OUTPUT_BASE = './results/openfold_scaling_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'

CONFIG = {
    'quick': {
        'n_sequences': 200,
        'seq_lengths': [100, 200, 400],
        'batch_size': 1,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_lengths': [100, 200, 300, 400, 500, 600, 700, 800],
        'batch_size': 1,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Sequences per length: {config['n_sequences']}")
print(f"Sequence lengths: {config['seq_lengths']}")
print(f"Bootstrap rounds: {config['n_bootstrap']}")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
# Generate Protein Sequences

import numpy as np

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')  # 20 standard amino acids

def generate_protein_sequences(n_sequences, seq_length, seed=320):
    """Generate random protein sequences."""
    rng = np.random.default_rng(seed)
    return [
        ''.join(rng.choice(AMINO_ACIDS, size=seq_length))
        for _ in range(n_sequences)
    ]

sequences_by_length = {}
for length in config['seq_lengths']:
    cache_path = f'{CACHE_DIR}/openfold_synth_seqs_L{length}.txt'
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            seqs = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(seqs)} cached sequences (L={length})")
    else:
        seqs = generate_protein_sequences(config['n_sequences'], length, seed=320 + length)
        with open(cache_path, 'w') as f:
            for s in seqs:
                f.write(s + '\n')
        print(f"Generated {len(seqs)} sequences (L={length}), cached")
    sequences_by_length[length] = seqs

total = sum(len(v) for v in sequences_by_length.values())
print(f"\nTotal: {total} sequences across {len(sequences_by_length)} lengths")
print(f"Sample (L=100): {sequences_by_length[config['seq_lengths'][0]][0][:50]}...")

In [ ]:
# Protein Perturbation Suite

from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


def mutate_protein(sequence, mutation_rate, rng):
    """Randomly substitute amino acids at the given rate."""
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [aa for aa in AMINO_ACIDS if aa != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_sequence(sequence):
    """Reverse amino acid order (tests order sensitivity)."""
    return sequence[::-1]


class ProteinPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_reverse=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_reverse = include_reverse

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"aa_sub_{int(rate * 100)}pct"
            perturbed = [mutate_protein(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='substitution', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'Random AA substitutions at {rate*100:.0f}% rate',
            )
        if self.include_reverse:
            results['reverse'] = PerturbedSet(
                name='reverse', category='reverse',
                sequences=[reverse_sequence(s) for s in sequences],
                params={}, description='Full sequence reversal',
            )
        return results

    def get_perturbation_names(self):
        names = [f"aa_sub_{int(r*100)}pct" for r in self.snp_rates]
        if self.include_reverse:
            names.append('reverse')
        return names

    def summary(self):
        lines = [
            'Protein Perturbation Protocol',
            '=' * 40,
            f'AA substitution rates: {self.snp_rates}',
            f'Sequence reversal: {self.include_reverse}',
            f'Total perturbation sets: {len(self.get_perturbation_names())}',
        ]
        return '\n'.join(lines)


suite = ProteinPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
print(suite.summary())

test_seqs = sequences_by_length[config['seq_lengths'][0]][:5]
test_perturbed = suite.run_all(test_seqs)
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(test_seqs, pset.sequences)
    ]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# OpenFold Embedding Function (SoloSeq + Evoformer)
#
# Pipeline per sequence:
#   1. ESM-1b last-layer embeddings (per-residue, dim=1280)
#   2. OpenFold Evoformer (SoloSeq mode, templates disabled)
#   3. Extract single representation s (per-residue, dim=384)
#   4. Mean-pool over residues -> (384,) embedding vector

import torch
import torch.nn.functional as F
import gc

# AlphaFold2 / OpenFold residue order
RESTYPES = 'ARNDCQEGHILKMFPSTWYV'
AA_TO_IDX = {aa: i for i, aa in enumerate(RESTYPES)}


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"  GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")


def make_openfold_embedding_fn(device='cuda', max_length=1022):
    """Load ESM-1b + OpenFold SoloSeq and return an embedding function.

    Returns (embed_fn, of_model, n_params_M) where embed_fn maps
    a list of AA strings to a (N, 384) numpy array.
    """
    import esm as esm_module
    from openfold.config import model_config
    from openfold.model.model import AlphaFold

    # --- Load ESM-1b ---
    print("Loading ESM-1b...")
    esm_model, esm_alphabet = esm_module.pretrained.esm1b_t33_650M_UR50S()
    esm_model = esm_model.eval().to(device)
    batch_converter = esm_alphabet.get_batch_converter()
    ESM_REPR_LAYER = 33
    esm_params = sum(p.numel() for p in esm_model.parameters()) / 1e6
    print(f"  ESM-1b loaded ({esm_params:.0f}M params)")

    # --- Load OpenFold SoloSeq ---
    print("Loading OpenFold SoloSeq (Evoformer)...")
    cfg = model_config("seq_model_esm1b_ptm", train=False)
    cfg.model.template.enabled = False          # skip templates for embedding extraction
    cfg.globals.use_flash = False
    cfg.globals.use_deepspeed_evo_attention = False

    of_model = AlphaFold(cfg)
    state = torch.load(WEIGHTS_PATH, map_location='cpu')
    of_model.load_state_dict(state, strict=False)  # strict=False: ignore template weights
    of_model = of_model.eval().to(device)
    of_params = sum(p.numel() for p in of_model.parameters()) / 1e6
    print(f"  OpenFold loaded ({of_params:.0f}M params, templates disabled)")

    if torch.cuda.is_available():
        print(f"  Total GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    # Hook to capture Evoformer single representation
    _captured = {}
    def _evo_hook(module, inp, out):
        m, z, s = out  # Evoformer returns (m, z, s)
        _captured['s'] = s.detach()
    of_model.evoformer.register_forward_hook(_evo_hook)

    @torch.no_grad()
    def embed(sequences):
        all_embeddings = []
        n = len(sequences)

        for idx, seq in enumerate(sequences):
            seq = seq[:max_length]
            L = len(seq)

            # --- Step 1: ESM-1b last-layer embeddings ---
            data = [("protein", seq)]
            _, _, tokens = batch_converter(data)
            tokens = tokens.to(device)
            esm_out = esm_model(tokens, repr_layers=[ESM_REPR_LAYER])
            esm_repr = esm_out['representations'][ESM_REPR_LAYER]
            seq_embedding = esm_repr[0, 1:L+1, :]  # (L, 1280), strip BOS/EOS

            # --- Step 2: Build OpenFold feature dict ---
            aatype = torch.tensor(
                [AA_TO_IDX.get(aa, 20) for aa in seq],
                dtype=torch.long, device=device,
            )
            target_feat = F.one_hot(aatype.clamp(max=21), num_classes=22).float()
            residue_index = torch.arange(L, dtype=torch.long, device=device)
            seq_mask = torch.ones(L, device=device, dtype=torch.float32)
            msa_mask = torch.ones(1, L, device=device, dtype=torch.float32)

            NUM_ITERS = 1  # no recycling for speed
            features = {
                'aatype':        aatype.unsqueeze(-1).expand(-1, NUM_ITERS),
                'target_feat':   target_feat.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
                'residue_index': residue_index.unsqueeze(-1).expand(-1, NUM_ITERS),
                'seq_embedding': seq_embedding.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
                'seq_mask':      seq_mask.unsqueeze(-1).expand(-1, NUM_ITERS),
                'msa_mask':      msa_mask.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
            }

            # --- Step 3: Forward pass ---
            _captured.clear()
            try:
                _ = of_model(features)
            except Exception:
                # Structure module may fail (needs atom features we don't provide).
                # The hook already captured the Evoformer output.
                pass

            # --- Step 4: Extract & mean-pool single representation ---
            if 's' in _captured:
                s = _captured['s'].squeeze(0)  # (L, 384)
                pooled = s.mean(dim=0).cpu().numpy()
            else:
                # Fallback: use ESM-1b last layer if Evoformer failed
                pooled = seq_embedding.mean(dim=0).cpu().numpy()

            all_embeddings.append(pooled)

            if (idx + 1) % 10 == 0 or idx == n - 1:
                print(f"    Sequence {idx+1}/{n}", end='\r')

        print()
        return np.stack(all_embeddings)

    return embed, of_model, of_params


print("OpenFold embedding wrapper ready")

In [ ]:
# Set Up Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,         # embeddings are already pooled
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import time
import pandas as pd
from evaluation_harness import ModelReport

print('=' * 70)
print(f'OPENFOLD SCALING EXPERIMENT -- Phase: {PHASE.upper()}')
print(f'Scaling variable: sequence length')
print('=' * 70)

# Load models once (shared across lengths)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_fn, of_model, n_params_m = make_openfold_embedding_fn(device=device)

reports = []
all_detailed_rows = []

for length_idx, seq_length in enumerate(config['seq_lengths']):
    print(f"\n{'='*70}")
    print(f"[{length_idx+1}/{len(config['seq_lengths'])}] Sequence length = {seq_length} AA")
    print('=' * 70)

    start_time = time.time()
    sequences = sequences_by_length[seq_length]
    model_label = f'OpenFold_L{seq_length}'

    # Perturbations for this length
    print('\nGenerating perturbations...')
    pert_suite = ProteinPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
    perturbed_sets = pert_suite.run_all(sequences)

    # Clean embeddings (with caching)
    safe_name = f'openfold_L{seq_length}'
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached clean embeddings: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} seqs, L={seq_length})...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    # Perturbed embeddings
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    # Shesha evaluation
    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_label,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': model_label,
            'seq_length': seq_length,
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted L={seq_length} in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Magnitude:     {summary["mean_perturbation_magnitude"]:.4f}')

# Save detailed CSV
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/openfold_scaling_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Compare & Save Results

from evaluation_harness import compare_models

print('Generating comparison...')
comparison = compare_models(reports, output_dir=RESULTS_DIR)

print('\n' + '=' * 70)
print('SUMMARY')
print('=' * 70 + '\n')

for model_name, data in comparison['models'].items():
    s = data['summary']
    print(f'{model_name}:')
    print(f'  Composite Stability:  {s["mean_composite_stability"]:.4f} +/- {s["std_composite_stability"]:.4f}')
    print(f'  RDM Similarity:       {s["mean_rdm_similarity_score"]:.4f} +/- {s["std_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Stability:      {s["mean_perturbation_stability_score"]:.4f} +/- {s["std_perturbation_stability_score"]:.4f}')
    print(f'  Pert. Magnitude:      {s["mean_perturbation_magnitude"]:.4f} +/- {s["std_perturbation_magnitude"]:.4f}')
    print()

In [ ]:
# Visualize -- Stability vs Sequence Length

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

summaries = [r.summary() for r in reports]
model_names = [r.model_name for r in reports]
lengths = config['seq_lengths']

metrics = {
    'mean_composite_stability':          ('Composite Stability', 'tab:blue'),
    'mean_rdm_similarity_score':         ('RDM Similarity (clean vs. perturbed)', 'tab:green'),
    'mean_perturbation_stability_score': ('Perturbation Direction Consistency', 'tab:purple'),
    'mean_perturbation_magnitude':       ('Perturbation Magnitude', 'tab:red'),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax_idx, (metric_key, (label, color)) in enumerate(metrics.items()):
    ax = axes[ax_idx]
    values = [s[metric_key] for s in summaries]
    std_key = metric_key.replace('mean_', 'std_')
    errors = [s.get(std_key, 0) for s in summaries]

    ax.errorbar(lengths, values, yerr=errors, fmt='o-', color=color,
                linewidth=2, markersize=10, capsize=5, capthick=2)
    ax.set_xlabel('Sequence Length (amino acids)')
    ax.set_ylabel(label)
    ax.set_title(label, fontweight='bold')
    ax.grid(True, alpha=0.3)

    for i, L in enumerate(lengths):
        ax.annotate(f'L={L}', (lengths[i], values[i]),
                    textcoords='offset points', xytext=(0, 12),
                    ha='center', fontsize=9, color='gray')

fig.suptitle(
    'OpenFold Evoformer: Geometric Stability vs. Sequence Length (SoloSeq)',
    fontsize=15, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/openfold_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{RESULTS_DIR}/openfold_scaling_{PHASE}.pdf', bbox_inches='tight')
plt.show()
print(f'Saved to {RESULTS_DIR}/openfold_scaling_{PHASE}.png and .pdf')

In [ ]:
# Per-Perturbation Breakdown

print('Per-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nDetailed CSV at: {detailed_path}')

In [ ]:
# Per-Perturbation Heatmap

import seaborn as sns

pivot = df_detailed.pivot_table(
    index='perturbation', columns='model',
    values='rdm_similarity', aggfunc='mean',
)

col_order = sorted(pivot.columns, key=lambda x: int(x.split('_L')[1]))
pivot = pivot[col_order]

fig, ax = plt.subplots(figsize=(max(8, len(col_order) * 2.5), 6))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=pivot.values.min() * 0.95, vmax=1.0,
    ax=ax, linewidths=0.5,
)
ax.set_title('RDM Similarity by Sequence Length and Perturbation',
             fontweight='bold')
ax.set_ylabel('Perturbation')
ax.set_xlabel('OpenFold @ Sequence Length (short \u2192 long)')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/openfold_heatmap_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved heatmap to {RESULTS_DIR}/openfold_heatmap_{PHASE}.png')